# apertus-eval-prep — ranking stability on Colab

Runtime → Change runtime type → **T4 GPU**.

1. Smoke (`stability_smoke.yaml`) — minutes, already on GitHub.
2. **Paper matrix** (`stability.yaml`, `--profile t4`) — 34 cells × 800 items. Hours; resume-safe.

Use a **separate** registry (`results/registry_paper.jsonl`) so the n=4 smoke does not mix into rankings.

In [ ]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))
if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")

## Paper matrix (T4)

34 cells. `--profile t4` skips 7B fp16, 7B int8, and 7B vLLM.
Re-run this cell after a disconnect; finished `config_hash` rows are skipped.

To finish across sessions, run **one model at a time** (uncomment one `--only-model` line).

In [ ]:
# All 34 T4 cells. Prefer this if the runtime will stay up for many hours.
!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml \
  --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl

# If Colab dies, rerun the same command. Or do one model per session:
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model HuggingFaceTB/SmolLM2-1.7B-Instruct
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model Qwen/Qwen2.5-3B-Instruct
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model microsoft/Phi-3.5-mini-instruct
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --out-dir results/runs --registry results/registry_paper.jsonl --only-model Qwen/Qwen2.5-7B-Instruct

In [ ]:
from google.colab import files
from pathlib import Path

!python -m apertus_eval_prep report --registry results/registry_paper.jsonl --out reports/stability_paper
!python -m apertus_eval_prep paper-tables --registry results/registry_paper.jsonl --out paper/_generated_tables.md
!zip -r paper_matrix_artifacts.zip results/runs results/registry_paper.jsonl reports/stability_paper paper/_generated_tables.md
print("zip bytes", Path("paper_matrix_artifacts.zip").stat().st_size)
files.download("paper_matrix_artifacts.zip")

Unpack `paper_matrix_artifacts.zip` into the Mac clone. Do not edit numbers. Commit `results/registry_paper.jsonl`, new `results/runs/*.json`, and `reports/stability_paper/`.